# 05 - System integration, deployment and external test (M8)
Packages detect -> crop -> classify -> segment -> verdict, exports to ONNX with parity checks, serves
over HTTP, benchmarks latency on the T4, evaluates on the team-captured photos, and fills the
acceptance-criteria table A1-A9.

In [ ]:
# Setup — mount Drive before resolving the repository.
import os, sys
from pathlib import Path
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    if not Path("/content/drive/MyDrive").is_dir():
        drive.mount("/content/drive")

def _resolve_spv_repo(search_root: Path, explicit: str | None = None) -> Path:
    def valid(path):
        return (path / "src" / "spv").is_dir() and (path / "configs" / "config.yaml").is_file()
    if explicit:
        candidate = Path(explicit).expanduser().resolve()
        if not valid(candidate):
            raise FileNotFoundError(f"SPV_REPO is not a project root: {candidate}")
        return candidate
    candidates = set()
    for pattern in ("smart-pet-vision", "*smart-pet-vision", "*/smart-pet-vision", "*/*/smart-pet-vision"):
        candidates.update(path.resolve() for path in search_root.glob(pattern) if valid(path))
    if valid(search_root):
        candidates.add(search_root.resolve())
    if len(candidates) != 1:
        raise FileNotFoundError(
            f"Expected one smart-pet-vision repository under {search_root}; found {len(candidates)}. "
            "Add the shared folder shortcut to My Drive, or set SPV_REPO to its exact path."
        )
    return candidates.pop()

if IN_COLAB:
    REPO = _resolve_spv_repo(Path("/content/drive/MyDrive"), os.environ.get("SPV_REPO"))
else:
    explicit = os.environ.get("SPV_REPO")
    local_root = next((path for path in (Path.cwd(), *Path.cwd().parents)
                       if (path / "src" / "spv").is_dir()), Path.cwd())
    REPO = _resolve_spv_repo(local_root, explicit)
os.chdir(REPO)
if IN_COLAB:
    !pip -q install -r requirements.txt
sys.path.insert(0, os.path.abspath("src"))

from spv.common import load_config, seed_everything, save_json, load_json
cfg = load_config()
if IN_COLAB:  # images on fast local disk (re-downloaded each session); results stay on Drive
    cfg["paths"]["raw"] = Path("/content/data/raw")
    cfg["paths"]["processed"] = Path("/content/data/processed")
seed_everything(cfg["seed"])
RUNS = Path(cfg["paths"]["runs"])
import torch; print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")

In [ ]:
from spv import data
df = data.ensure_data(cfg)          # first run: download + clean + split (about 10 min)
IMG = data.dataset_dir(cfg) / "images"
PROC = Path(cfg["paths"]["processed"])
print(len(df), "images after cleaning")

In [ ]:
from spv.pipeline import Manifest, PetPipeline
import pandas as pd
cls_table = pd.read_csv(RUNS / "classification" / "comparison.csv")
best = cls_table.sort_values("val_macro_f1", ascending=False).iloc[0]   # chosen on validation, not test
op = load_json(RUNS / "detection" / "operating_point.json")
seg = load_json(RUNS / "segmentation" / "weights.json")
pc = cfg["pipeline"]
manifest = Manifest(detector=op["weights"], segmenter=seg["weights"],
                    classifier=str(RUNS / "classification" / f"{best.model}.pt"),
                    classifier_kind=best.model, temperature=float(best.temperature),
                    detector_conf=op["conf"], detector_iou=op["nms_iou"],
                    verify_threshold=pc["verify_threshold"], min_mask_fraction=pc["min_mask_fraction"],
                    image_size=cfg["classification"]["image_size"], imgsz=cfg["detection"]["imgsz"],
                    segmenter_conf=pc["segmenter_conf"], segmenter_iou=pc["segmenter_iou"],
                    mask_match_iou=pc["mask_match_iou"])
manifest.save(RUNS / "manifest.json")
pipe = PetPipeline(manifest, cfg)

## Demo

In [ ]:
import base64, io
from PIL import Image
from IPython.display import display
sample = df[df.split == "test"].sample(3, random_state=1)
for name in sample.image:
    r = pipe.analyze(Image.open(IMG / f"{name}.jpg"))
    print(name, r["verdict"], r["species"], round(r["confidence"], 3), [x["code"] for x in r["reasons"]])
    display(Image.open(io.BytesIO(base64.b64decode(r["annotated_png"]))).resize((320, 320)))

## ONNX export and numerical parity (A9)

In [ ]:
from spv import deploy, classify
DEP = RUNS / "deploy"; DEP.mkdir(parents=True, exist_ok=True)
cls_model = classify.load_model(best.model, cfg, Path(manifest.classifier))
cls_onnx = deploy.export_classifier_onnx(cls_model, DEP / "classifier.onnx", manifest.image_size)
det_onnx = deploy.export_yolo_onnx(Path(manifest.detector), manifest.imgsz)
parity = [deploy.classifier_parity(cls_model, cls_onnx, manifest.image_size),
          deploy.yolo_parity(Path(manifest.detector), det_onnx, manifest.imgsz)]
save_json(parity, DEP / "onnx_parity.json"); parity

In [ ]:
# The service runs the detector from ONNX
manifest_onnx = Manifest(**{**manifest.__dict__, "detector": str(det_onnx)})
manifest_onnx.save(RUNS / "manifest_onnx.json")
pipe_onnx = PetPipeline(manifest_onnx, cfg)

## Latency benchmark on the T4 (A8)

In [ ]:
bench_imgs = [IMG / f"{n}.jpg" for n in df[df.split == "test"].image.head(50)]
bench = {"pytorch": deploy.benchmark(pipe, bench_imgs, warmup=10, runs=200),
         "onnx_detector": deploy.benchmark(pipe_onnx, bench_imgs, warmup=10, runs=200)}
save_json(bench, DEP / "benchmark.json"); bench

## HTTP service (A7)

In [ ]:
import os
os.environ["SPV_MANIFEST"] = str(RUNS / "manifest.json")
from fastapi.testclient import TestClient
from spv import serve
serve.pipeline.cache_clear()  # reload the explicitly selected PyTorch manifest
client = TestClient(serve.app)
print(client.get("/health").json())
print("non-image upload ->", client.post("/analyze", files={"file": ("x.txt", b"x", "text/plain")}).status_code)
svc = deploy.service_success_rate(client, [IMG / f"{n}.jpg" for n in df[df.split == "test"].image.head(300)])
save_json(svc, DEP / "service.json"); svc

To run the service outside the notebook:
```
SPV_MANIFEST=runs/manifest.json PYTHONPATH=src uvicorn spv.serve:app --port 8000
curl -F "file=@photo.jpg" "http://localhost:8000/analyze?images=false"
```

## External test on team-captured photos (bonus)
Put the phone photos in `data/team/raw/` and a `data/team/labels.csv` with columns `image,species`
(image = file name without extension). Optional: CVAT masks as PNG in `data/team/masks/`.
The photos are never used for training or threshold selection.

In [ ]:
TEAM = Path(cfg["paths"]["team"])
if (TEAM / "raw").exists() and (TEAM / "labels.csv").exists():
    team_meta = deploy.prepare_team(TEAM / "raw", TEAM / "images", df)
    print("Possible copies of public images:", int(team_meta.possible_public_duplicate.sum()))
    masks_dir = TEAM / "masks" if (TEAM / "masks").exists() else None
    team_summary, team_rows = deploy.evaluate_team(pipe, TEAM / "images", TEAM / "labels.csv", masks_dir)
    team_rows.to_csv(DEP / "team_predictions.csv", index=False)
    save_json(team_summary, DEP / "team_metrics.json"); print(team_summary)
else:
    n_raw = len(list((TEAM / "raw").glob("*.jpg"))) if (TEAM / "raw").exists() else 0
    print(f"Team set not evaluated yet: {n_raw} photos, human labels (labels.csv) pending annotation")

## Acceptance criteria A1-A9

In [ ]:
cls = load_json(RUNS / "classification" / f"{best.model}_test_metrics.json")["test_calibrated"]
det = load_json(RUNS / "detection" / "detection_test_metrics.json")
segm = load_json(RUNS / "segmentation" / "yolo_seg_test_metrics.json")
rows = [
 ("A1", "Classification accuracy", cls["accuracy"], ">= 0.90", cls["accuracy"] >= 0.90),
 ("A2", "Classification macro-F1", cls["macro_f1"], ">= 0.90", cls["macro_f1"] >= 0.90),
 ("A3", "Detection mAP50", det["coco"]["mAP50"], ">= 0.75", det["coco"]["mAP50"] >= 0.75),
 ("A4", "Detection recall at operating point", det["test_at_operating_point"]["recall"], ">= 0.80",
  det["test_at_operating_point"]["recall"] >= 0.80),
 ("A5", "Segmentation mean IoU", segm["mIoU"], ">= 0.75", segm["mIoU"] >= 0.75),
 ("A6", "Segmentation mean Dice", segm["mean_dice"], ">= 0.85", segm["mean_dice"] >= 0.85),
 ("A7", "Service success rate", svc["success_rate"], ">= 0.99", svc["passed_A7"]),
 ("A8", "p95 latency, T4, batch 1 (ms)", bench["pytorch"]["latency_ms"]["p95"], "<= 2000",
  bench["pytorch"]["passed_A8"]),
 ("A9a", "ONNX max abs diff (classifier)", parity[0]["max_abs_diff"], "<= 0.001", parity[0]["passed_A9"]),
 ("A9b", "ONNX max abs diff (detector, pixel outputs)", parity[1]["max_abs_diff"], "<= 0.001", parity[1]["passed_A9"]),
]
acc = pd.DataFrame(rows, columns=["id", "criterion", "value", "target", "met"])
acc.to_csv(RUNS / "acceptance.csv", index=False); acc

## Deployment evidence (added 24 Oct 2026)
The cells above served the pipeline with the ONNX detector through FastAPI's in-process `TestClient`, and the ONNX
backend ran on ONNX Runtime's CPU provider although the benchmark recorded the T4. This section replaces that
evidence: (1) the service is started with uvicorn in a separate process and called over real HTTP, with the
deployed PyTorch manifest; (2) each backend is benchmarked in its own process, so peak memory belongs to that backend,
and the detector's execution provider is recorded; (3) ONNX parity is also measured on real test images; (4) the
acceptance table is recomputed with A7 and A8 measured on the deployed PyTorch backend; (5) the team-set evaluation
is rerun after the four label corrections of 19 Oct; (6) referrals and multi-animal photos are analysed; (7) a
system-level failure gallery is drawn. Benchmark images: the first 50 test images, cycled, batch 1, 10 warm-up
and 200 timed runs.

In [ ]:
import json
import pandas as pd
from spv import deploy
from spv.pipeline import Manifest, PetPipeline
DEP = RUNS / "deploy"
MAN_PT, MAN_ONNX = RUNS / "manifest.json", RUNS / "manifest_onnx.json"
test_imgs = [IMG / f"{n}.jpg" for n in df[df.split == "test"].image]
svc = deploy.http_service_test(REPO, MAN_PT, test_imgs[:300], log_path=DEP / "uvicorn_log.txt")
save_json(svc, DEP / "service_http.json")
print(json.dumps(svc, indent=1, default=float))

In [ ]:
bench_imgs = test_imgs[:50]
b_pt = deploy.benchmark_isolated(REPO, MAN_PT, bench_imgs, DEP / "benchmark_pytorch.json")
b_onnx = deploy.benchmark_isolated(REPO, MAN_ONNX, bench_imgs, DEP / "benchmark_onnx_detector.json")
table = pd.DataFrame({k: {"detector EP": v["detector_execution_provider"], "p50 ms": v["latency_ms"]["p50"],
                          "p95 ms": v["latency_ms"]["p95"], "p99 ms": v["latency_ms"]["p99"],
                          "img/s": v["throughput_img_per_s"], "peak GPU MB (PyTorch allocator)": v["peak_gpu_mem_mb"],
                          "peak RSS MB (own process)": v["peak_process_rss_mb"]}
                      for k, v in {"pytorch": b_pt, "onnx_detector": b_onnx}.items()}).T
table.round(1)

In [ ]:
op = load_json(RUNS / "detection" / "operating_point.json")
par = deploy.yolo_parity_real(Path(Manifest.load(MAN_PT).detector), Path(Manifest.load(MAN_ONNX).detector),
                              test_imgs[:50], cfg["detection"]["imgsz"], op["conf"], op["nms_iou"])
save_json(par, DEP / "onnx_parity_real_images.json")
par

In [ ]:
best = pd.read_csv(RUNS / "classification" / "comparison.csv").sort_values("val_macro_f1", ascending=False).iloc[0]
cls = load_json(RUNS / "classification" / f"{best.model}_test_metrics.json")["test_calibrated"]
det = load_json(RUNS / "detection" / "detection_test_metrics.json")
segm = load_json(RUNS / "segmentation" / "yolo_seg_test_metrics.json")
parity = load_json(DEP / "onnx_parity.json")
rows = [
 ("A1", "Classification accuracy", cls["accuracy"], ">= 0.90", cls["accuracy"] >= 0.90),
 ("A2", "Classification macro-F1", cls["macro_f1"], ">= 0.90", cls["macro_f1"] >= 0.90),
 ("A3", "Detection mAP50", det["coco"]["mAP50"], ">= 0.75", det["coco"]["mAP50"] >= 0.75),
 ("A4", "Detection recall at operating point", det["test_at_operating_point"]["recall"], ">= 0.80",
  det["test_at_operating_point"]["recall"] >= 0.80),
 ("A5", "Segmentation mean IoU", segm["mIoU"], ">= 0.75", segm["mIoU"] >= 0.75),
 ("A6", "Segmentation mean Dice", segm["mean_dice"], ">= 0.85", segm["mean_dice"] >= 0.85),
 ("A7", "Service success rate (uvicorn over HTTP, PyTorch backend)", svc["success_rate"], ">= 0.99", svc["passed_A7"]),
 ("A8", "p95 latency, T4, batch 1, PyTorch backend (ms)", b_pt["latency_ms"]["p95"], "<= 2000", b_pt["passed_A8"]),
 ("A9a", "ONNX max abs diff (classifier)", parity[0]["max_abs_diff"], "<= 0.001", parity[0]["passed_A9"]),
 ("A9b", "ONNX max abs diff (detector, pixel outputs)", parity[1]["max_abs_diff"], "<= 0.001", parity[1]["passed_A9"]),
]
acc = pd.DataFrame(rows, columns=["id", "criterion", "value", "target", "met"])
acc.to_csv(RUNS / "acceptance.csv", index=False)
acc

In [ ]:
pipe = PetPipeline(Manifest.load(MAN_PT), cfg)
TEAM = Path(cfg["paths"]["team"])
team_summary, team_rows = deploy.evaluate_team(pipe, TEAM / "images", TEAM / "labels.csv", TEAM / "masks")
saved = load_json(DEP / "team_metrics.json")
prev = pd.read_csv(DEP / "team_predictions.csv")
same_metrics = all(abs(team_summary[k] - saved[k]) < 1e-9 for k in team_summary)
same_rows = prev[["image", "pred", "verdict", "reasons"]].fillna("").equals(
    team_rows[["image", "pred", "verdict", "reasons"]].fillna(""))
print(json.dumps(team_summary, indent=1))
print("identical to team_metrics.json:", same_metrics, "| identical predictions and verdicts:", same_rows)

In [ ]:
lab = pd.read_csv(TEAM / "labels.csv")[["image", "n_animals"]]
t = team_rows.merge(lab, on="image")
t["reasons"] = t.reasons.fillna("")
t["n_reasons"] = t.reasons.apply(lambda s: len([x for x in s.split(";") if x]))
multi = t[t.n_animals > 1]
analysis = {
    "photos": int(len(t)), "referred": int((t.verdict != "ready").sum()),
    "reason_counts": t.reasons.str.split(";").explode().loc[lambda s: s != ""].value_counts().to_dict(),
    "referred_with_more_than_one_reason": int((t.n_reasons > 1).sum()),
    "multi_animal_photos": int(len(multi)),
    "multi_animal_flagged_multiple": int(multi.reasons.str.contains("multiple_animals").sum()),
    "multi_animal_accepted_ready": int((multi.verdict == "ready").sum()),
    "single_animal_flagged_multiple": int(((t.n_animals == 1) & t.reasons.str.contains("multiple_animals")).sum()),
    "no_animal_referrals": int(t.reasons.str.contains("no_animal").sum()),
    "classification_errors": int(((t.true != t.pred) & t.true.isin(["cat", "dog"])).sum()),
    "thresholds": {"detector_conf": pipe.m.detector_conf, "nms_iou": pipe.m.detector_iou,
                   "verify_threshold": pipe.m.verify_threshold, "min_mask_fraction": pipe.m.min_mask_fraction,
                   "detector_imgsz": pipe.m.imgsz, "classifier_imgsz": pipe.m.image_size},
}
save_json(analysis, DEP / "team_referral_analysis.json")
print(json.dumps(analysis, indent=1, default=float))

In [ ]:
err = t[(t.true != t.pred) & t.true.isin(["cat", "dog"])].copy()
err["statement"] = err.apply(lambda r: f"true {r.true}, predicted {r.pred} with confidence {r.confidence:.2f}; "
                             + ("accepted - error escaped review" if r.verdict == "ready" else f"referred ({r.reasons})"), axis=1)
noa = t[t.reasons.str.contains("no_animal")].copy()
noa["statement"] = noa.apply(lambda r: f"{r.n_animals} animal(s) in the photo, detector found none; referred", axis=1)
mis = t[(t.n_animals > 1) & (t.verdict == "ready")].sort_values("n_animals", ascending=False).head(8).copy()
mis["statement"] = mis.apply(lambda r: f"{r.n_animals} animals, detector found {r.n_detected}; accepted as ready - "
                             "multiple-animal rule missed", axis=1)
cases = pd.concat([err, noa, mis])[["image", "statement"]]
cases.to_csv(DEP / "system_failures.csv", index=False)
deploy.system_failure_gallery(pipe, TEAM / "images", cases, DEP / "figures" / "system_failures.jpg")
from IPython.display import Image as Show, display
display(Show(str(DEP / "figures" / "system_failures.jpg"), width=900))
print(len(cases), "cases")

In [ ]:
from PIL import Image as PILImage
fpb = pd.read_csv(RUNS / "detection" / "fp_breakdown.csv")
names = fpb[fpb.kind.isin(["class_confusion", "localisation_and_class"])].image.unique()
truth = df.set_index("image").species
rows = []
for n in names:
    r = pipe.analyze(PILImage.open(IMG / f"{n}.jpg"), with_images=False)
    rows.append({"image": n, "true": truth[n], "detector_top": r["detections"][0]["species"] if r["detections"] else None,
                 "classifier": r["species"], "verdict": r["verdict"],
                 "reasons": ";".join(x["code"] for x in r["reasons"])})
cc = pd.DataFrame(rows)
cc.to_csv(DEP / "class_confusion_pipeline.csv", index=False)
cc